# Dataset Exploration: Camargo 2021 & Reznick 2021

This notebook inspects the benchmark datasets used for Shadow Limb trajectory prediction.

**Goals:**
1. Load and inspect the Camargo 2021 `.mat` files (columns, sampling rates, units)
2. Visualize IMU signals alongside ankle angle ground truth over gait cycles
3. Confirm IMU channel mapping matches our pipeline expectations
4. Check for missing data, outliers, and sampling rate consistency
5. Survey terrain conditions (level ground, ramps, stairs)

In [ ]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.io import loadmat

sys.path.insert(0, os.path.join(os.getcwd(), ".."))
from src.trajectory import config

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")
%matplotlib inline

print(f"Camargo dir: {config.CAMARGO_DIR}")
print(f"Reznick dir: {config.REZNICK_DIR}")

## 1. Camargo 2021 - File Discovery

The dataset should be organized as `data/camargo2021/AB01/`, `AB02/`, ..., `AB22/`,
each containing `.mat` files for different locomotion conditions.

In [ ]:
mat_files = sorted(glob.glob(os.path.join(config.CAMARGO_DIR, "**", "*.mat"), recursive=True))
print(f"Total .mat files found: {len(mat_files)}")

if not mat_files:
    print("\nNo data found! Run: python scripts/download_datasets.py")
    print("for download instructions.")
else:
    # Show first few files
    for f in mat_files[:10]:
        print(f"  {os.path.relpath(f, config.CAMARGO_DIR)}")
    if len(mat_files) > 10:
        print(f"  ... and {len(mat_files) - 10} more")

## 2. Inspect .mat File Structure

Load one file and print all variable names, shapes, and types.
This tells us the exact column names for IMU channels and ankle angles.

In [ ]:
if mat_files:
    sample_file = mat_files[0]
    print(f"Inspecting: {os.path.relpath(sample_file, config.CAMARGO_DIR)}\n")
    
    data = loadmat(sample_file, squeeze_me=True)
    
    print("Top-level keys:")
    for key in sorted(data.keys()):
        if key.startswith("__"):
            continue
        val = data[key]
        if hasattr(val, "shape"):
            print(f"  {key:40s}  shape={str(val.shape):20s}  dtype={val.dtype}")
        elif hasattr(val, "dtype") and val.dtype.names:
            print(f"  {key:40s}  structured array, fields={val.dtype.names}")
        else:
            print(f"  {key:40s}  type={type(val).__name__}")
else:
    print("No .mat files to inspect.")

## 3. Deep-Dive: Nested Structure

Camargo .mat files often have nested structures. Let's recursively explore.

In [ ]:
def explore_mat(obj, prefix="", max_depth=3, depth=0):
    """Recursively print the structure of a loaded .mat object."""
    if depth > max_depth:
        return
    
    if isinstance(obj, dict):
        for k, v in sorted(obj.items()):
            if k.startswith("__"):
                continue
            explore_mat(v, prefix=f"{prefix}.{k}" if prefix else k, max_depth=max_depth, depth=depth+1)
    elif hasattr(obj, "dtype") and obj.dtype.names:
        print(f"{prefix}  [structured] fields={obj.dtype.names}")
        for name in obj.dtype.names:
            try:
                child = obj[name]
                if hasattr(child, "item"):
                    child = child.item()
                explore_mat(child, prefix=f"{prefix}.{name}", max_depth=max_depth, depth=depth+1)
            except Exception:
                pass
    elif isinstance(obj, np.ndarray):
        if obj.ndim == 0:
            inner = obj.item()
            if hasattr(inner, "dtype") and inner.dtype.names:
                explore_mat(inner, prefix=prefix, max_depth=max_depth, depth=depth)
            else:
                print(f"{prefix}  scalar -> {type(inner).__name__}")
        else:
            print(f"{prefix}  ndarray shape={obj.shape} dtype={obj.dtype}")
    else:
        print(f"{prefix}  {type(obj).__name__}")

if mat_files:
    explore_mat(data, max_depth=4)

## 4. Identify IMU and Joint Angle Columns

Based on the structure above, update the column names in `src/trajectory/config.py`
if they differ from the defaults. The key columns we need:

| Purpose | Expected Pattern | Config Variable |
|---------|-----------------|----------------|
| Shank accelerometer | `shank_Accel_X/Y/Z` | `SHANK_IMU_COLS` |
| Shank gyroscope | `shank_Gyro_X/Y/Z` | `SHANK_IMU_COLS` |
| Thigh accelerometer | `thigh_Accel_X/Y/Z` | `THIGH_IMU_COLS` |
| Thigh gyroscope | `thigh_Gyro_X/Y/Z` | `THIGH_IMU_COLS` |
| Ankle angle (target) | `ankle_angle_r` | `TARGET_COL` |

In [ ]:
# After inspecting the structure above, try to extract data columns.
# This cell adapts to whatever structure you find.

if mat_files:
    # Attempt to find column-like keys containing IMU or angle data
    imu_keywords = ["accel", "gyro", "imu", "shank", "thigh"]
    angle_keywords = ["ankle", "angle", "joint", "kinematics"]
    
    all_keys = [k for k in data.keys() if not k.startswith("__")]
    
    print("Potential IMU-related keys:")
    for k in all_keys:
        if any(kw in k.lower() for kw in imu_keywords):
            val = data[k]
            shape = val.shape if hasattr(val, "shape") else "N/A"
            print(f"  {k}  shape={shape}")
    
    print("\nPotential joint angle keys:")
    for k in all_keys:
        if any(kw in k.lower() for kw in angle_keywords):
            val = data[k]
            shape = val.shape if hasattr(val, "shape") else "N/A"
            print(f"  {k}  shape={shape}")

## 5. Visualize a Single Gait Trial

Plot the shank IMU signals (accel + gyro) aligned with the ankle angle ground truth.

In [ ]:
# This cell will work once you've confirmed the column names from above.
# Update the variable names below to match what the .mat files actually contain.

def plot_gait_trial(trial_data, title="Gait Trial", sample_rate=200):
    """
    Plot IMU signals and ankle angle for a trial.
    trial_data: dict with keys for each signal channel (numpy arrays).
    """
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    
    # Determine time axis
    n_samples = None
    for v in trial_data.values():
        if isinstance(v, np.ndarray) and v.ndim == 1:
            n_samples = len(v)
            break
    if n_samples is None:
        print("Could not determine signal length.")
        return
    
    t = np.arange(n_samples) / sample_rate
    
    # Accelerometer
    ax = axes[0]
    for col_name in ["shank_Accel_X", "shank_Accel_Y", "shank_Accel_Z"]:
        if col_name in trial_data:
            ax.plot(t, trial_data[col_name], label=col_name, alpha=0.8)
    ax.set_ylabel("Acceleration (m/s²)")
    ax.set_title(f"{title} — Shank Accelerometer")
    ax.legend(loc="upper right")
    
    # Gyroscope
    ax = axes[1]
    for col_name in ["shank_Gyro_X", "shank_Gyro_Y", "shank_Gyro_Z"]:
        if col_name in trial_data:
            ax.plot(t, trial_data[col_name], label=col_name, alpha=0.8)
    ax.set_ylabel("Angular Velocity (rad/s)")
    ax.set_title("Shank Gyroscope")
    ax.legend(loc="upper right")
    
    # Ankle angle
    ax = axes[2]
    target_key = config.TARGET_COL
    if target_key in trial_data:
        ax.plot(t, trial_data[target_key], color="crimson", linewidth=2, label=target_key)
    ax.set_ylabel("Ankle Angle (deg)")
    ax.set_xlabel("Time (s)")
    ax.set_title("Ankle Dorsiflexion / Plantarflexion (Ground Truth)")
    ax.legend(loc="upper right")
    
    plt.tight_layout()
    plt.show()

print("plot_gait_trial() defined. Use it once you extract trial data arrays.")

## 6. Sampling Rate Verification

Confirm the IMU data is sampled at ~200 Hz.

In [ ]:
# If the .mat files contain a time vector, use it to verify sampling rate.
# Otherwise, check the paper: Camargo 2021 reports 200 Hz for IMU.

if mat_files:
    time_keys = [k for k in data.keys() if "time" in k.lower() or k == "t"]
    if time_keys:
        for tk in time_keys:
            t_arr = data[tk].flatten()
            if len(t_arr) > 2:
                dt = np.median(np.diff(t_arr))
                fs = 1.0 / dt if dt > 0 else 0
                print(f"  {tk}: median dt={dt:.6f}s -> fs={fs:.1f} Hz  (n={len(t_arr)} samples)")
    else:
        print("No explicit time vector found. Camargo 2021 paper states 200 Hz for IMU.")

## 7. Terrain Condition Survey

List the locomotion conditions available per subject.

In [ ]:
if mat_files:
    conditions = {}
    for f in mat_files:
        parts = os.path.relpath(f, config.CAMARGO_DIR).split(os.sep)
        subject = parts[0] if len(parts) > 1 else "unknown"
        trial_name = os.path.splitext(parts[-1])[0]
        conditions.setdefault(subject, []).append(trial_name)
    
    print(f"Subjects found: {len(conditions)}")
    for subj in sorted(conditions.keys())[:3]:
        trials = sorted(conditions[subj])
        print(f"\n  {subj}: {len(trials)} trials")
        for t in trials[:8]:
            print(f"    - {t}")
        if len(trials) > 8:
            print(f"    ... and {len(trials) - 8} more")

## 8. Data Quality Checks

In [ ]:
if mat_files:
    print("Checking first file for NaNs and outliers...\n")
    for key in sorted(data.keys()):
        if key.startswith("__"):
            continue
        val = data[key]
        if isinstance(val, np.ndarray) and val.dtype.kind == "f":
            nan_count = np.isnan(val).sum()
            if nan_count > 0:
                print(f"  {key}: {nan_count} NaN values ({nan_count/val.size*100:.1f}%)")
    print("\nDone. If no NaN messages printed, data is clean.")

## 9. Next Steps

After running this notebook:

1. **Update `src/trajectory/config.py`** with the correct column names if they differ from defaults
2. **Run the data pipeline**: `src/trajectory/dataset.py` will use these column names to build training windows
3. **Train the model**: `python -m src.trajectory.train`

Key things to record:
- Actual IMU column names: `____________`
- Actual ankle angle column name: `____________`
- Confirmed sampling rate: `____________ Hz`
- Number of usable trials per subject: `____________`